In [0]:
# Gold layer: fact_web_logs (Kimball transaction-grain fact - web events)
# Source: b2b_ecommerce.silver.logs - the web-server-log stream, one row
#         per log_id (AUTO CDC dedup, latest wins - silver owns validation).
# Grain:  exactly one row per log_id (the web-event grain) - the MERGE
#         key, so retries and re-runs are idempotent.
# Joins:  date_sk = dim_date's smart YYYYMMDD key from event_time
#         (pre-flight 2026-11-19: all 538,152 event_times valid - 0
#         nulls, 2024-10-29..2026-11-06, 100% inside dim_date's range
#         - no validity flag needed, unlike fact_order_items'
#         order_date). customer_sk is the conversion bridge for the
#         web KPIs (KPI 9 device-specific conversion): a web log's
#         auth_user is the logged-in customer's email, and dim_customers
#         carries email exactly for this match. dim_customers is SCD1
#         (current state, one row per customer) and pre-flight verified
#         ALL 1,581 distinct auth_users match exactly one dim email (0
#         unmatched, 0 duplicate emails = 0 multi-match), so a plain
#         LEFT join on email is lossless and needs no as-of logic.
#         214,131 authenticated rows resolve to a customer; the 324,021
#         anonymous rows keep their row with NULL customer_sk - facts
#         retain rows, they never drop them. auth_user itself is then
#         excluded (company_name precedent in fact_leads: customer_sk
#         supersedes the raw bridge input; the dim owns emails).
# Columns: log_id (business + MERGE key), date_sk, raw event_time (ART
#         wall clock, same clock as orders.created_at - kept because the
#         sessionization enrichment keys on request timestamps),
#         customer_sk (conversion bridge), client_ip (input for the
#         IP-geo enrichment via ref_ip_geo - KPI 10 geographic web
#         traffic), user_agent (input for the device-classification
#         enrichment - KPI 9 desktop/mobile/tablet), method + endpoint +
#         status_code (KPI 11 web activity quality: 4xx/5xx error rates
#         by endpoint), referer (requirements' Source 2 lists "Referer &
#         path (for traffic analysis)" - KPI 10's top traffic sources).
#         Bots (~22% bot-like user agents: UptimeRobot, AhrefsBot, ...)
#         are RETAINED - the fact is gross, exactly like fact_order_items
#         keeps cancelled orders; bot exclusion is a view/enrichment
#         policy, not a fact-load decision. Excluded per the KPI audit
#         (no KPI uses them; one-line additions if reports later need
#         them): query_string, protocol, response_bytes, auth_user
#         (superseded by customer_sk), source_file (lineage), updated_at
#         (gold timestamp policy).
# Storage: Delta-backed STREAMING TABLE (facts are stored tables, not
#         views), loaded via AUTO CDC FROM SNAPSHOT - the same
#         ingestion-time pattern as the gold dims, fact_order_items and
#         fact_leads.
# MERGE / idempotency: keyed on log_id (requirements' MERGE
#         constraint - safe to retry without duplicates). The snapshot
#         view is recomputed each run; an unchanged source diffs to zero
#         events and changes nothing. Type 1 - facts do not version
#         history; the dims own that.
# Quality: no expectations - AUTO CDC FROM SNAPSHOT flows cannot carry
#         them (same decision as the dims and the other facts), and
#         silver already owns row-level validation.
# Layout: partitioned by date_sk (requirements: fact_* partitioned by
#         date). Delta does not allow clustering keys alongside
#         partition columns
#         (SPECIFY_CLUSTER_BY_WITH_PARTITIONED_BY_IS_NOT_ALLOWED), so no
#         cluster_by - same decision as fact_order_items and fact_leads;
#         the 739 date partitions prune MERGE scans and KPI queries.
from pyspark import pipelines as dp
from pyspark.sql import functions as F

GOLD = "b2b_ecommerce.gold"
SILVER_LOGS = "b2b_ecommerce.silver.logs"


# The complete fact snapshot, recomputed at each pipeline run: the
# silver web-log stream, customer_sk-resolved via the email bridge,
# trimmed to the KPI-required columns. A batch temporary view - the
# engine reads it at each run (ingestion-time snapshot versioning). No
# driver-side actions.
@dp.temporary_view(name="fact_web_logs_snapshot")
def fact_web_logs_snapshot():
    # The conversion bridge: auth_user -> the customer's dim row. SCD1
    # dim (one row per customer, unique email - pre-flight verified 0
    # duplicates), so a plain LEFT join resolves exactly one
    # customer_sk per authenticated row; anonymous rows pass through
    # with NULL customer_sk (rows retained).
    customers = (
        spark.read.table(f"{GOLD}.dim_customers")
        .where(F.col("email").isNotNull())
        .select("email", "customer_sk")
    )

    return (
        spark.read.table(SILVER_LOGS)
        .join(
            customers,
            F.col("auth_user") == F.col("email"),
            "left",
        )
        .withColumn(
            "date_sk",
            (
                F.year("event_time") * 10000
                + F.month("event_time") * 100
                + F.dayofmonth("event_time")
            ).cast("int"),
        )
        .select(
            "log_id",
            "date_sk",
            "event_time",
            "customer_sk",
            "client_ip",
            "user_agent",
            "method",
            "endpoint",
            "status_code",
            "referer",
        )
    )


# Delta-backed streaming table: the AUTO CDC target. Partitioned by the
# date key (requirements: fact_* partitioned by date). No clustering
# keys: Delta forbids CLUSTER BY together with PARTITIONED BY.
dp.create_streaming_table(
    name="fact_web_logs",
    comment=(
        "Kimball transaction-grain fact: one row per log_id from the silver "
        "web-log stream. date_sk = YYYYMMDD from event_time (all dates valid, "
        "2024-10-29..2026-11-06, fully covered by dim_date). customer_sk bridges "
        "an authenticated request to its buyer-side customer via auth_user = "
        "dim_customers.email (SCD1, unique - all 1,581 distinct auth_users "
        "match exactly one customer; anonymous requests are retained with NULL "
        "customer_sk) and is the device-to-conversion linkage for the web KPIs. "
        "client_ip and user_agent are kept raw as the inputs for the gold IP-geo "
        "and device-classification enrichments; method/endpoint/status_code drive "
        "the error-rate-by-endpoint KPI; referer drives top-traffic-source "
        "analysis; event_time (ART wall clock) is kept for the sessionization "
        "enrichment. Bot traffic is retained (the fact is gross - exclusion is a "
        "view policy). MERGE-loaded via AUTO CDC FROM SNAPSHOT keyed on log_id "
        "(idempotent retries). Partitioned by date_sk (requirements: fact_* "
        "partitioned by date)."
    ),
    partition_cols=["date_sk"],
)


# The auto-merge load: diff each run's fact snapshot against the last
# committed one and apply inserts/updates/deletes in place (Type 1 -
# facts do not version history; the dims own that).
dp.create_auto_cdc_from_snapshot_flow(
    target="fact_web_logs",
    source="fact_web_logs_snapshot",  # ingestion-time snapshot versioning
    keys=["log_id"],
    stored_as_scd_type=1,
)